# Chapter 12. CNN으로 이미지 이해하기


## 1. 컴퓨터에게 사진은 숫자의 모음이다

사진을 아주 크게 확대하면 작은 점들이 보인다. 이 점 하나를 **픽셀**이라고 한다. 컴퓨터는 각 픽셀의 밝기나 색을 숫자로 저장한다. 따라서 컴퓨터가 사진을 이해한다는 것은 많은 숫자 속에서 의미 있는 규칙을 찾는 일이다.

흑백 사진은 보통 픽셀마다 밝기 값 하나를 가진다. 컬러 사진은 빨강, 초록, 파랑의 세 가지 값을 사용한다. 각각의 정보를 담는 통로를 **채널**이라고 한다. RGB 사진은 색 정보가 담긴 세 장의 얇은 종이를 겹쳐 놓은 것처럼 생각할 수 있다.

PyTorch에서 CNN에 넣는 이미지 묶음은 보통 ‘사진 수, 채널 수, 높이, 너비’의 순서로 정리한다. 예를 들어 컬러 사진 열 장을 넣으면 채널 수는 3이다. 채널이 마지막에 있는 이미지 자료도 있으므로, 모델이 기대하는 순서로 바꾸는 과정이 필요하다.

숫자를 0에서 255 사이로 저장한 사진을 0에서 1 사이로 바꾸는 등 값의 크기도 맞춘다. 다만 이미 학습된 모델을 사용할 때는 그 모델이 학습할 때 사용한 전처리 기준을 따라야 한다.


## 2. cnn

일반적인 완전연결 신경망은 한 층의 각 뉴런을 다음 층의 모든 뉴런과 연결한다. 사진을 길게 펼쳐 넣으면 많은 연결이 필요하고, 서로 가까운 픽셀들이 이웃이라는 구조를 모델이 직접 활용하기 어렵다.

사진에서는 가까운 점들이 함께 선이나 모서리를 만드는 경우가 많다. 고양이의 귀가 사진 왼쪽에 있든 오른쪽에 있든 비슷한 모양이라는 사실도 중요하다. **CNN은 가까운 부분을 함께 보고, 같은 특징 찾기 방법을 여러 위치에서 반복해서 사용한다.**

예를 들어 세로선을 찾는 방법을 하나 배웠다면 사진의 왼쪽에서도, 가운데에서도, 오른쪽에서도 활용할 수 있다. 위치마다 완전히 새로운 방법을 배울 필요가 줄어드는 것이다.

이처럼 연결을 작은 영역으로 제한하고 같은 가중치를 재사용하면, 비슷한 규모의 완전연결 층보다 학습할 숫자를 크게 줄일 수 있다. 사진의 공간 구조를 활용한다는 점이 CNN의 중요한 장점이다. 다만 사진을 얼마나 옮겨도 결과가 무조건 같다는 뜻은 아니다.


## 3. 합성곱은 작은 창으로 사진을 훑는 계산이다

**합성곱 층**은 작은 창을 사진 위에서 옮겨 가며 특징을 찾는다. 이 창에 들어 있는 학습 가능한 숫자 묶음을 **필터** 또는 **커널**이라고 한다. 예를 들어 3×3 크기라면 가로 세 칸, 세로 세 칸의 작은 영역을 본다.

한 위치에서는 창 안의 이미지 값과 필터의 값을 같은 자리끼리 곱하고 더한다. 이렇게 얻은 숫자가 그 위치의 출력이 된다. 그다음 창을 옮겨 같은 계산을 반복한다. 자를 대고 종이를 조금씩 훑으며 특정 무늬가 있는지 확인하는 모습과 비슷하다.

하나의 필터로 사진 전체를 훑어 얻은 결과를 **특징 맵**이라고 한다. 어떤 특징이 어느 위치에서 얼마나 강하게 나타나는지 표시한 지도라고 이해했다.

필터가 여러 개라면 특징 지도도 여러 개 만들어진다. 어떤 필터는 선이나 경계에, 다른 필터는 다른 무늬에 반응할 수 있다. 실제 학습에서는 사람이 모든 필터를 직접 설계하지 않고, 정답을 잘 맞히도록 필터의 숫자가 바뀐다.

일반적인 합성곱에서 컬러 사진용 필터는 입력의 세 채널을 모두 살펴본다. 채널별로 계산한 내용을 합쳐 하나의 출력 채널을 만든다. 그래서 필터의 가로·세로 크기뿐 아니라 입력 채널 수도 가중치 개수에 영향을 준다.


## 4. 층을 쌓으면 더 넓은 모양을 볼 수 있다

첫 합성곱 층은 작은 영역의 단순한 특징을 찾는다. 뒤쪽 층은 앞에서 만든 특징 맵을 입력으로 받아 특징들을 조합한다. 선들이 모여 모서리나 무늬를 이루고, 이런 정보가 모여 더 큰 형태를 구분하는 데 쓰일 수 있다.

특정 출력 하나에 영향을 줄 수 있는 원래 이미지의 영역을 **수용 영역**이라고 한다. 처음에는 작은 부분을 보더라도 여러 층을 거치면 더 넓은 영역의 정보가 모인다. 작은 글자 조각을 보고 단어를 파악한 다음 문장의 뜻을 이해하는 과정에 비유할 수 있다.

다만 모든 뉴런이 사람이 이름 붙일 수 있는 특징 하나만 담당하는 것은 아니다. ‘앞에서는 단순한 특징, 뒤에서는 더 복잡한 조합’이라는 흐름으로 이해하는 것이 좋다.

합성곱 뒤에는 보통 ReLU 같은 활성화 함수를 넣는다. 11강에서 정리했듯, 이런 비선형 변화가 있어야 여러 층을 통해 복잡한 관계를 표현할 수 있다.


## 5. 스트라이드와 패딩

**스트라이드**는 필터를 한 번에 몇 칸 옮길지 정하는 값이다. 스트라이드가 1이면 한 칸씩 촘촘하게 보고, 2이면 두 칸씩 옮긴다. 크게 이동하면 계산할 위치가 줄어 출력 이미지도 작아지는 경향이 있다. 대신 세밀한 위치 정보를 놓칠 수 있다.

**패딩**은 이미지 가장자리에 여분의 칸을 붙이는 것이다. 대표적으로 0을 붙이는 zero padding을 사용한다. 필터가 가장자리까지 계산하도록 돕고 출력 크기를 조절하는 데 쓰인다.

가로와 세로가 각각 다섯 칸인 입력에 3×3 필터를 한 칸씩 움직인다고 생각해 보았다. 패딩이 없으면 필터가 들어갈 위치는 가로와 세로 각각 세 곳이다. 가장자리에 한 칸씩 붙이면 각각 다섯 곳에서 계산할 수 있어 원래 크기를 유지한다.

패딩을 쓰지 않는 방식을 valid라고 부르기도 한다. same은 출력 크기를 입력과 맞추려는 패딩 방식과 관련된 표현이다. 다만 구현마다 지원하는 스트라이드 조건이 다르다. 원본의 PyTorch 예제에서는 스트라이드가 1일 때 `padding="same"`을 쓰고, 스트라이드가 2일 때는 패딩 칸 수를 직접 지정한다.

0을 붙이는 일도 결과에 영향을 준다. 실제 사진에 없던 어두운 테두리를 추가하는 셈이어서 가장자리의 반응이 달라질 수 있다. 패딩은 단순히 빈 공간을 붙이는 일이 아니라 계산 조건을 바꾸는 선택이다.


## 6. 합성곱의 가중치

합성곱의 가중치 개수는 필터의 크기, 입력 채널 수, 출력 채널 수에 따라 정해진다. 사진이 커진다고 같은 필터의 가중치가 늘어나는 것은 아니다. 같은 도구를 더 많은 위치에 반복해서 사용하는 것이기 때문이다.

예를 들어 RGB 사진에 3×3 필터 하나를 적용하면 세 채널에 각각 아홉 개씩, 총 27개의 가중치가 필요하다. 출력에 더하는 편향 하나까지 사용하면 28개다. 이런 필터를 100개 쓰면 학습할 값은 2,800개가 된다.

앞 층에서 특징 맵 100개를 만들었다면 다음 층의 입력 채널은 100개다. 이때 3×3 필터 하나에는 가중치 900개가 필요하다. 입력 채널이 처음의 RGB 세 개에 고정되는 것이 아니라, 앞 층의 출력 채널 수를 따라간다는 점이 중요하다.

사진이 커지면 가중치 수는 같아도 계산 위치와 특징 맵의 크기가 늘어난다. 그래서 필요한 계산량과 메모리는 커질 수 있다. 가중치가 적다는 말과 메모리를 거의 쓰지 않는다는 말은 다르다.


## 7. 풀링으로 특징 지도를 줄이기

**풀링**은 작은 영역의 값들을 하나의 대표값으로 줄이는 방법이다. 큰 지도의 세부 정보를 조금 덜어 내고 중요한 표시를 남기는 것과 비슷하다.

**최대 풀링**은 영역 안에서 가장 큰 값을 고른다. 예를 들어 네 값이 1, 3, 2, 4라면 4만 남긴다. 강하게 반응한 특징을 남기는 방식이라고 이해했다. 2×2 영역을 겹치지 않게 묶으면 가로와 세로 크기가 각각 대략 절반으로 줄어든다.

**평균 풀링**은 영역 안의 평균값을 남긴다. 위의 네 값에서는 2.5가 된다. 가장 강한 반응 하나보다 영역 전체의 평균적인 반응을 요약한다.

일반적인 최대 풀링과 평균 풀링은 대표값을 고르는 규칙이 정해져 있으므로 학습하는 가중치가 없다. 필터의 숫자를 학습하는 합성곱과 다른 점이다. 다만 원래 LeNet의 풀링처럼 추가 학습 파라미터가 있는 역사적인 변형은 별도로 구분해야 한다.

풀링은 계산량을 줄이고 작은 위치 변화에 덜 민감하게 만드는 데 도움이 될 수 있다. 하지만 정확히 어디에 특징이 있었는지는 일부 잃어버린다. 그래서 물체 경계를 픽셀 단위로 찾아야 할 때는 줄어든 위치 정보를 어떻게 보완할지 생각해야 한다.


## 8. 채널 방향 풀링과 전역 평균 풀링

보통 풀링은 이미지의 가로·세로 방향으로 작은 영역을 묶는다. 원본에는 **채널 방향 풀링**도 나온다. 같은 위치에 있는 여러 채널의 값을 묶어 대표값을 고르는 방법이다. 공간을 줄이는지, 채널을 줄이는지 구분해야 한다.

**전역 평균 풀링**은 각 특징 맵 전체의 평균을 구한다. 특징 맵이 100장이면 각 장에서 숫자 하나씩, 총 100개의 요약값을 얻는다. 특정 특징이 사진 전체에서 어느 정도 나타났는지를 요약하는 것이다.

원본에서는 `AdaptiveAvgPool2d(1)`을 이용해 각 특징 맵을 1×1 크기로 만든다. 앞에서 나온 특징 맵의 가로·세로가 달라도 채널마다 숫자 하나로 요약할 수 있다.

특징 맵의 모든 칸을 길게 펼쳐 큰 완전연결 층에 넣는 대신 이런 요약을 사용하면, 뒤쪽 층의 파라미터를 줄일 수 있다. 다만 평균 하나만으로는 특징의 정확한 위치를 알 수 없다는 한계도 있다.


## 9. CNN으로 옷 사진 분류하기

원본은 Fashion MNIST의 옷 사진을 CNN으로 분류한다. 사진은 28×28 크기의 흑백 이미지이고, 열 가지 종류 중 하나를 맞히는 과제다.

모델 앞쪽에서는 합성곱과 ReLU로 특징을 찾는다. 중간의 풀링은 특징 맵을 줄이고, 다음 합성곱은 앞의 특징들을 더 복잡하게 조합한다. 마지막에는 특징을 펼쳐 완전연결 층에 전달하고 열 개의 점수를 출력한다.

출력 점수는 아직 확률로 바꾸기 전의 값인 로짓이다. 열 개 중 가장 높은 점수를 가진 종류를 예측 답으로 고를 수 있다. 학습에 사용하는 `CrossEntropyLoss`에는 이 로짓을 직접 넣는다.

원본 구조에서는 28칸이 풀링을 거치며 14칸, 7칸, 3칸으로 줄어든다. 마지막 채널 수가 256이므로 펼친 숫자는 256×3×3, 즉 2,304개다. 층을 바꿀 때는 다음 층이 받을 숫자의 개수도 함께 확인해야 한다.

Dropout은 특정 특징에 지나치게 의존하지 않도록 돕는다. 11강에서 배운 과적합 방지 방법이 CNN 학습에도 연결된다는 점을 확인했다.


## 10. CNN 구조가 발전해 온 방향

**LeNet-5**는 숫자 이미지 인식과 연결되는 대표적인 초기 CNN이다. 합성곱으로 특징을 찾고, 풀링으로 줄이고, 마지막에 종류를 판단하는 흐름을 보여 준다. 원래 구조에는 오늘날 흔한 CNN과 다른 활성화 함수나 출력 방식도 있으므로, 기본 흐름을 중심으로 이해했다.

**AlexNet**은 더 크고 깊은 CNN을 활용했다. 합성곱을 한 번 할 때마다 반드시 풀링을 넣는 대신 합성곱 층을 연속으로 쌓는 구조도 사용했다.

**GoogLeNet**은 Inception 모듈을 도입했다. 서로 다른 크기의 특징을 살펴보는 계산들을 여러 갈래에서 수행한 뒤 결과를 모은다. 같은 사진을 작은 창과 큰 창으로 함께 살펴보는 방식에 비유할 수 있다. 계산을 효율적으로 만들기 위해 채널 수를 줄이는 처리도 활용한다.

**SENet**은 특징 맵마다 중요도를 조절한다. 모든 특징을 똑같이 다루기보다 현재 입력에서 도움이 되는 채널에 더 큰 비중을 주는 방식이다.

**EfficientNet**은 모델의 깊이, 채널 수로 표현되는 너비, 입력 이미지 크기를 균형 있게 늘리는 방법을 다룬다. 한쪽만 무조건 키우기보다 주어진 계산 자원에 맞춰 함께 조절한다.

**ConvNeXt**는 다른 현대적인 이미지 모델에서 활용한 설계 아이디어를 반영해 CNN 구조를 발전시킨 모델이다. 원본에서는 이미 학습된 ConvNeXt를 불러와 예측과 전이학습에 사용한다.


## 11. Xception과 깊이별 분리 합성곱

일반 합성곱은 주변 위치의 정보와 여러 채널의 정보를 함께 계산한다. **깊이별 분리 합성곱**은 이 일을 두 단계로 나누는 방법이다. Xception은 이 아이디어를 활용하는 대표적인 구조다.

먼저 각 입력 채널을 따로 살펴보며 가로·세로의 무늬를 찾는다. 이 단계를 depthwise convolution이라고 한다. 원본 구현은 채널마다 별도의 공간 필터를 적용한다.

그다음 1×1 합성곱으로 같은 위치에 있는 여러 채널의 정보를 섞는다. 이를 pointwise convolution이라고 한다. 1×1은 한 칸만 본다는 뜻이지 아무 계산도 하지 않는다는 뜻이 아니다. 그 한 위치에 쌓인 여러 채널을 조합하고 출력 채널 수를 바꿀 수 있다.

예를 들어 여러 사람이 각자 한 종류의 자료를 정리한 뒤, 그 결과를 모아 판단하는 모습과 비슷하다. 주변 무늬를 찾는 일과 채널들을 섞는 일을 나누어 계산량과 파라미터를 줄일 수 있다.

이것은 채널 방향 풀링과 다르다. 풀링은 정해진 규칙으로 대표값을 고르고, 깊이별 분리 합성곱은 두 단계의 필터를 학습한다.


## 12. ResNet은 입력이 지나갈 지름길을 만든다

신경망을 깊게 만들면 더 복잡한 것을 표현할 수 있지만, 실제로 학습시키기는 어려워질 수 있다. **ResNet**은 이 문제를 줄이기 위해 중간 계산을 건너뛰는 연결을 둔다. 이를 skip connection이라고 한다.

입력을 여러 층에서 계산한 결과에 원래 입력을 더해 다음으로 보낸다. 처음부터 완전히 새로운 답을 만드는 대신 기존 정보에 무엇을 보태거나 고칠지 배우는 방식이다. 잘 작성된 글을 전부 다시 쓰기보다 필요한 부분을 수정하는 상황과 비슷하다.

이 구조에서 새로 배우는 부분을 **잔차**라고 한다. 입력이 직접 전달되는 경로는 정보와 기울기가 흐르는 데 도움을 준다. 그래서 매우 깊은 모델도 학습하기 쉬워질 수 있다.

두 값을 더하려면 모양이 맞아야 한다. 중간에 이미지 크기나 채널 수가 달라졌다면 지름길 쪽도 1×1 합성곱 등으로 맞춘다. 크기가 다른 두 배열을 무조건 더할 수 있는 것은 아니다.

원본에서는 이런 잔차 블록을 반복해서 ResNet-34 구조를 만든다. 마지막에는 전역 평균 풀링으로 특징을 요약하고 분류 층에서 답을 고른다.


## 13. 이미 학습된 모델을 불러와 사용하기

**사전 학습 모델**은 많은 이미지로 이미 훈련된 모델이다. 사진의 선, 무늬, 형태 등을 찾는 데 유용한 가중치가 들어 있다. 이를 불러오면 모든 것을 처음부터 배울 필요를 줄일 수 있다.

원본은 ImageNet으로 학습된 ConvNeXt를 사용한다. 모델뿐 아니라 함께 제공되는 전처리 방식도 불러온다. 사진 크기나 색 값의 기준이 학습 당시와 크게 다르면 모델이 기대하는 입력과 달라질 수 있기 때문이다.

예측할 때는 모델을 평가 모드로 바꾸고 불필요한 기울기 계산을 끈다. 출력된 점수에서 가장 큰 값을 고르면 예측 종류를 얻을 수 있고, 상위 여러 개를 확인하면 어떤 후보를 높게 평가했는지 볼 수 있다.

확률을 볼 때는 전체 클래스 점수를 기준으로 확률로 바꾼 뒤 상위 결과를 고르는 것이 중요하다. 상위 세 점수만 따로 뽑아 확률로 바꾸면 그 세 후보끼리 합이 100%가 되므로, 전체 클래스에 대한 원래 확률과 달라진다. 원본의 상위 후보 예제를 읽을 때 이 차이를 구분했다.


## 14. 꽃 분류에 전이학습 적용하기

원본에서는 사전 학습된 모델을 이용해 Flowers102의 꽃 종류 102개를 구분한다. 기존 모델이 찾을 수 있는 이미지 특징을 활용하되, 마지막 답을 고르는 부분을 꽃 분류에 맞게 바꾸는 것이다.

처음에는 기존 특징 추출 부분의 가중치를 동결하고 새 분류 부분을 학습한다. 기존에 배운 특징을 바로 크게 바꾸기보다, 그 특징으로 새 문제를 풀도록 마지막 판단부터 적응시키는 과정이다.

그다음 필요하면 동결을 풀고 기존 부분도 조금씩 조정한다. 이것이 미세 조정이다. 11강에서 공부한 전이학습이 이미지 문제에서 어떻게 쓰이는지 보여 주는 예제다.

새로운 사진에서도 잘 맞히는지 검증 데이터로 확인하고, 마지막 평가에는 테스트 데이터를 사용한다. 훈련 정확도만 높아졌다는 이유로 새 과제에 잘 적응했다고 판단하면 안 된다.


## 15. 데이터 증강은 같은 사진을 다양하게 보여 주는 것이다

**데이터 증강**은 훈련 사진을 조금씩 바꿔 모델에게 보여 주는 방법이다. 좌우를 뒤집거나, 조금 회전하거나, 일부를 잘라 크기를 맞추거나, 밝기를 바꿀 수 있다.

꽃을 한 방향에서 찍은 사진만 계속 보여 주면 모델이 그 방향이나 배경에 지나치게 익숙해질 수 있다. 여러 모습으로 보여 주면 같은 종류의 꽃이 다양한 상황에서 나타날 수 있다는 점을 학습하도록 돕는다.

중요한 것은 바꾼 뒤에도 정답이 유지되어야 한다는 점이다. 꽃 사진의 작은 회전은 괜찮을 수 있지만, 숫자를 크게 회전시키면 다른 숫자처럼 보일 수 있다. 많이 바꾸는 것보다 문제에 맞게 바꾸는 것이 중요하다.

증강은 주로 훈련 데이터에 적용한다. 검증과 테스트에서는 비교가 가능하도록 일정한 전처리를 사용한다. 데이터 증강은 사진 수를 다양하게 보이게 하는 방법이지, 검증·테스트 자료를 훈련에 섞어도 된다는 뜻은 아니다.


## 16. 분류와 위치 찾기는 무엇이 다를까?

**이미지 분류**는 사진에 무엇이 있는지 답하는 과제다. **위치 찾기**는 그 대상이 사진의 어디에 있는지도 답한다. 예를 들어 ‘장미다’에서 한 걸음 더 나아가 장미를 둘러싼 사각형을 그리는 것이다.

이 사각형을 **경계 상자**, 영어로 bounding box라고 한다. 사각형은 왼쪽 위와 오른쪽 아래 좌표로 나타낼 수도 있고, 중심 위치와 가로·세로 길이로 나타낼 수도 있다. 같은 네 숫자라도 어떤 표현 방식인지에 따라 뜻이 달라진다.

원본의 FlowerLocator는 공통으로 추출한 특징을 두 갈래로 보낸다. 한쪽은 꽃의 종류를 예측하고, 다른 쪽은 상자를 나타내는 네 값을 예측한다. 하나의 모델이 두 가지 관련된 일을 함께 하는 구조다.

학습하려면 꽃 이름뿐 아니라 실제 위치를 표시한 정답 상자도 필요하다. 원본은 상자 두 개를 붙인 자료와 학습 준비 흐름을 보여 주며, 꽃 전체의 위치를 학습한 완성 결과를 제시하는 것은 아니다. 모델 구조를 만들었다는 것과 정확한 위치를 찾도록 학습했다는 것을 구분해야 한다.


## 17. 사진을 바꾸면 정답 위치도 함께 바꿔야 한다

물체 위치를 학습할 때 사진만 좌우로 뒤집고 정답 상자를 그대로 두면 문제가 생긴다. 실제 꽃은 오른쪽으로 옮겨 갔는데 정답은 왼쪽을 가리킬 수 있기 때문이다.

그래서 사진을 자르거나 회전하거나 뒤집을 때는 경계 상자도 같은 변환을 적용해야 한다. 원본에서는 이미지와 상자를 함께 처리하는 변환 도구를 사용한다.

회전한 상자를 다시 가로·세로에 평행한 사각형으로 감싸면 상자가 커질 수 있다. 기울어진 책을 책상 위에 놓고 그 전체를 수평 사각형으로 둘러싸면 빈 모서리 공간까지 포함되는 것과 비슷하다.

따라서 좌표가 계산되었다고 끝내기보다 실제 사진에 상자를 그려 보는 것이 중요하다. 정답 표시가 어긋나면 모델이 잘못된 위치를 배우게 된다.


## 18. 여러 물체를 찾는 객체 탐지

**객체 탐지**는 한 사진 속 여러 물체를 찾아 각각의 종류와 위치를 예측하는 과제다. 도로 사진에서 자동차, 사람, 신호등을 각각 찾는 일을 생각하면 된다. 대표 대상 하나의 종류와 위치만 찾는 문제보다 할 일이 많다.

원본은 학습된 YOLO 모델을 이용해 축구 사진과 교통 사진에서 물체를 찾는다. 결과에는 물체의 종류, 예측 점수, 경계 상자 등이 들어간다. 그 결과를 사진에 표시하면 어떤 물체를 어디서 찾았는지 확인할 수 있다.

탐지 모델은 같은 물체 주변에 여러 상자를 예측할 수 있다. 중복 상자를 정리하는 대표적인 방법이 **비최대 억제**, 즉 NMS다. 점수가 높은 상자를 남기고 많이 겹치는 같은 종류의 후보를 줄이는 방식이다. 어떤 후처리를 쓰는지는 모델에 따라 다르다.

예측 점수가 높다고 반드시 정답이라는 뜻은 아니다. 물체 종류뿐 아니라 상자가 실제 위치에 잘 맞는지도 평가해야 한다.


## 19. 탐지 결과는 어떻게 평가할까?

**IoU**는 예측 상자와 정답 상자가 얼마나 겹치는지 나타내는 값이다. 두 상자가 겹친 넓이를 두 상자를 합쳐 덮는 전체 넓이로 나눈다. 겹침이 없으면 0이고 완전히 같으면 1이다.

예를 들어 합쳐 덮는 넓이가 100이고 겹친 넓이가 60이면 IoU는 0.6이다. 객체 탐지에서는 보통 종류가 맞는지와 IoU가 정해진 기준을 넘는지를 함께 살펴본다. 같은 정답 물체를 중복해서 찾는 것도 모두 정답으로 세지는 않는다.

**정밀도**는 모델이 찾았다고 한 것 중 실제로 맞는 비율이다. 사람이라고 찾은 열 개 중 여덟 개가 맞으면 정밀도는 80%다. **재현율**은 실제로 있는 대상을 얼마나 많이 찾았는지다. 사람이 열 명 있는데 여섯 명을 찾으면 재현율은 60%다.

점수가 높은 결과만 남기면 잘못 찾은 것을 줄일 수 있지만 실제 물체를 놓칠 수도 있다. 반대로 기준을 낮추면 더 많이 찾는 대신 틀린 후보도 늘어날 수 있다. 그래서 한 가지 기준의 정확도만으로 탐지 성능을 설명하기 어렵다.

**AP**는 예측 점수의 기준을 바꾸며 얻는 정밀도와 재현율 관계를 요약한 값이다. **mAP**는 여러 종류의 AP를 평균한 지표다. 평가 방식에 따라 여러 IoU 기준도 함께 평균하므로, mAP 숫자를 비교할 때는 같은 평가 조건인지 확인해야 한다.


## 20. 영상 속 같은 물체를 따라가는 추적

**객체 추적**은 영상의 여러 장면에서 같은 물체를 계속 연결하는 일이다. 한 장면에서 자동차를 찾는 것이 탐지라면, 다음 장면의 자동차가 방금 본 그 자동차인지 알아보는 것이 추적이다.

영상은 연속된 사진인 프레임으로 구성된다. 추적에서는 물체마다 번호를 붙이고 장면이 바뀌어도 같은 대상의 번호를 유지하려고 한다. 원본은 차량 영상에 탐지와 추적을 적용해 각 물체의 track ID를 확인한다.

같은 ‘자동차’라는 종류 이름과 같은 ‘자동차 한 대’를 가리키는 추적 번호는 다르다. 사진 속 자동차가 다섯 대라면 모두 종류는 같지만 각각 다른 번호가 필요하다.

물체가 가려지거나 서로 비슷하게 생겼거나 빠르게 움직이면 연결이 어려워질 수 있다. 추적 번호가 있다고 해서 언제나 정확하게 같은 물체를 따라간다고 보장되는 것은 아니다.


## 21. 픽셀마다 종류를 정하는 시맨틱 분할

**시맨틱 분할**은 사진의 픽셀 하나하나에 종류를 붙이는 과제다. 사진 전체에 ‘사람’이라는 답 하나를 붙이거나 사각형을 그리는 데서 더 나아가, 사람에 해당하는 픽셀을 구체적으로 구분한다. 색칠 공부에서 영역마다 다른 색을 칠하는 모습과 비슷하다.

예를 들어 도로 사진에서 각 픽셀을 도로, 사람, 자동차, 하늘로 나눌 수 있다. 원본은 사전 학습된 FCN 모델을 사용해 사진에서 사람일 가능성이 높은 영역을 확인한다.

이때 만들어지는 **마스크**는 각 위치가 특정 대상에 해당하는지 표시하는 지도다. 확률 마스크라면 각 픽셀이 사람일 가능성을 연속적인 값으로 나타낼 수 있고, 그 값을 원본 사진과 결합해 사람 부분을 강조할 수 있다.

시맨틱 분할은 같은 종류의 서로 다른 대상을 반드시 따로 구분하지는 않는다. 사람이 두 명 있어도 두 사람 모두 ‘사람’이라는 같은 종류로 표시한다. 각각의 사람을 별도의 영역으로 구분하는 것은 인스턴스 분할과 연결되는 문제다.


## 22. FCN과 위치 정보 복원

**FCN**은 완전 합성곱 신경망이라는 뜻이다. 이미지 전체를 하나의 분류 답으로만 바꾸기보다, 공간 구조가 남아 있는 상태로 위치별 예측을 만드는 데 활용된다. 전형적으로 고정된 크기의 완전연결 분류 부분을 합성곱 방식으로 바꾼다.

완전연결 층은 정해진 개수의 입력을 기대하지만, 합성곱은 같은 필터를 위치마다 적용한다. 그래서 전체 구조가 허용하는 범위에서 여러 가로·세로 크기의 이미지를 처리하기 유리하다. 다만 실제 모델의 최소 크기나 전처리 조건은 확인해야 한다.

분할에서 어려운 점은 특징을 찾는 동안 이미지 크기를 줄여 놓았다는 것이다. 작은 지도만 남으면 물체가 대략 어디에 있는지는 알아도 정확한 경계를 그리기 어렵다.

이를 보완하려면 작은 특징 맵을 다시 키우는 업샘플링이 필요하다. 하지만 작아진 사진을 크게 늘리는 것만으로 사라진 세부 정보가 모두 돌아오지는 않는다. 더 앞쪽의 세밀한 특징을 함께 활용하는 등의 방법이 도움이 된다.

분류에서는 정확한 위치를 조금 잃어도 괜찮을 수 있지만, 픽셀별 답을 만드는 분할에서는 위치를 살리는 일이 핵심이라는 점을 이해했다.


## 23. CNN 학습에 메모리가 많이 필요한 이유

CNN은 가중치만 저장하는 것이 아니다. 각 층에서 만든 특징 맵도 메모리에 들어간다. 학습 중에는 나중에 역전파로 기울기를 계산할 때 필요한 중간값을 보관해야 하므로 예측만 할 때보다 메모리가 많이 필요할 수 있다.

한 번에 많은 사진을 넣거나, 사진 크기가 크거나, 채널 수가 많으면 특징 맵도 커진다. 모델 파일이 작다고 학습 중 사용하는 메모리까지 작다고 단정하면 안 된다.

메모리가 부족할 때는 먼저 한 번에 넣는 사진 수인 배치 크기를 줄이는 방법을 생각할 수 있다. 입력 크기나 채널 수, 층 수를 줄이거나 더 일찍 공간 크기를 줄이는 방법도 있지만, 세부 정보와 성능에 영향을 줄 수 있다.

일부 계산을 더 적은 비트의 숫자로 처리하는 방법도 있다. 숫자 하나가 차지하는 공간을 줄이는 대신 계산의 안정성과 지원 환경을 함께 고려해야 한다. 여러 장치에 모델을 나누는 방법도 가능하지만 구현이 더 복잡해진다.


## 24. 연습문제 1~7 정리

### 1. 이미지 분류에서 CNN의 장점은 무엇일까?

가까운 픽셀을 함께 보고 같은 필터를 여러 위치에서 사용한다. 그래서 이미지의 공간 구조를 활용하고, 위치가 달라도 비슷한 특징을 찾을 수 있다. 모든 위치를 별도로 연결하는 방식에 비해 파라미터를 줄이는 데도 도움이 된다.

### 2. 파라미터 수와 메모리는 어떻게 계산할까?

문제에서는 RGB 입력에 3×3 필터를 사용하고 세 합성곱 층의 출력 채널이 각각 100, 200, 400개다. 편향을 포함하면 첫 층은 2,800개, 둘째 층은 180,200개, 셋째 층은 720,400개로 합계 903,400개다.

메모리는 파라미터뿐 아니라 특징 맵도 세어야 한다. 문제의 패딩 가정에서 200×300 입력을 스트라이드 2로 줄이면 공간 크기는 100×150, 50×75, 25×38이 된다. 32비트 숫자 하나를 4바이트로 계산하면 각 층의 특징 맵은 사진 한 장당 약 6MB, 3MB, 1.52MB다.

앞 층의 메모리를 바로 재사용할 수 있다고 단순화하면, 예측 시 두 특징 맵과 가중치만으로 약 12.6MB가 필요하다. 학습 배치가 50장일 때는 세 층의 특징 맵 526MB, 입력 36MB, 가중치 약 3.6MB를 합쳐 약 565.6MB다. 원본은 중간값을 반올림해 약 564.6MB로 설명한다. 두 값 모두 기울기, 옵티마이저 상태, 계산용 임시 공간 등을 충분히 포함한 실제 사용량은 아니다. 메모리 계산의 목적은 크기가 무엇에 따라 늘어나는지 이해하는 데 있다.

### 3. GPU 메모리가 부족하면 어떻게 할까?

배치 크기를 줄이거나 모델과 특징 맵의 크기를 줄일 수 있다. 낮은 정밀도의 계산이나 여러 장치 활용도 후보가 된다. 다만 정보 손실과 학습 안정성에 영향을 줄 수 있으므로 각 방법의 대가를 함께 봐야 한다.

### 4. 최대 풀링도 가중치를 학습할까?

일반적인 최대 풀링은 가장 큰 값을 고르는 규칙이 정해져 있어 학습할 파라미터가 없다. 합성곱은 필터의 가중치를 학습한다는 점에서 다르다.

### 5. 주요 CNN은 어떤 아이디어를 추가했을까?

AlexNet은 더 크고 깊은 구조, GoogLeNet은 여러 갈래의 Inception 모듈, ResNet은 건너뛰기 연결, SENet은 채널 중요도 조절과 연결해서 기억했다. Xception은 깊이별 분리 합성곱, EfficientNet은 크기를 균형 있게 늘리는 방법, ConvNeXt는 현대적인 설계를 반영한 CNN으로 정리했다.

### 6. FCN은 어떤 점이 유용할까?

고정된 길이로 펼친 입력에만 의존하지 않고 공간별 계산을 유지할 수 있다. 구조가 허용하는 다양한 이미지 크기를 처리하고, 위치가 중요한 탐지나 분할에 활용하기 좋다.

### 7. 시맨틱 분할이 어려운 이유는 무엇일까?

앞에서 특징을 요약하며 줄인 위치 정보를 다시 세밀한 픽셀 예측으로 연결해야 하기 때문이다. 어떤 대상인지 아는 것과 그 대상의 정확한 경계를 그리는 것은 다른 문제다.
